In [ ]:
# This is first Simple, Naive RAG that I am building
# The pipeline is given below
# 1. Chunking Fixed Size Non Overlapping
# 2. Using Sentence-Embeddings (Optional Raw Bert Token Embeddings then pooling them via different pooling algos)
# 3. Storing these embeddings in a list or dict {token : embedding vector}
# 4. Vectorize the Query via Sentence-Embedding (Optional Raw Bert Token Embeddings then pooling)
# 5. Performing Cosine Similarity search
# 6. Retriving the top K documents
# 7. Using GPT or any other Generation Model

In [ ]:
# Corpus that I am using is a Claude Generated Document that has 5 topics each of 4 paragraphs
# Topics present in the document:
# 1. Photosynthesis (light-dependent reactions, Calvin cycle, C4/CAM pathways)
# 2. French Revolution (1789 origins, storming of the Bastille, Reign of Terror, Napoleon's rise)
# 3. HTTP/HTTPS (methods, status codes, statelessness, cookies, TLS encryption)
# 4. Return/refund policy (standard returns, final sale exclusions, defective items, exchanges)
# 5. Black holes (event horizon, Schwarzschild radius, supermassive black holes, Hawking radiation)

In [ ]:
# First Obtained the Multi Line string as a corpus where each line in VScode String is a single paragraph and for V1 RAG I am trrating
# Every Paragraph as a Chunk that will be embedded via SBERT
# Use split function on single new line as separator and obtain the list which also contains spaces
# To remove them simply sort and slice the docs list to obtain the actual doc level list where each element is a str and is a doc
# And each doc is a chunk and hence will be embedded accordingly
# There are 5 topics each having 4 paras so len(pre_pro_docs) = 20

In [1]:
!pip install sentence-transformers

In [2]:
corpus = """
Photosynthesis is the process by which plants convert light energy into chemical energy. It occurs primarily in the chloroplasts, specifically within structures called thylakoids. Chlorophyll, the green pigment in plants, absorbs light most efficiently in the blue and red wavelengths while reflecting green light. The overall reaction converts carbon dioxide and water into glucose and oxygen using energy from sunlight.
The light-dependent reactions of photosynthesis take place across the thylakoid membrane. Water molecules are split in a process called photolysis, releasing oxygen as a byproduct. This stage produces ATP and NADPH, which are then used to power the second stage of photosynthesis. Without sufficient light intensity, the rate of these reactions slows significantly, limiting overall plant growth.
The Calvin cycle, also called the light-independent reactions, occurs in the stroma of the chloroplast. Carbon dioxide is fixed into organic molecules using the ATP and NADPH generated earlier. This cycle does not directly require light, but depends entirely on the products of the light-dependent reactions to function. The end product of the Calvin cycle is glucose, which the plant uses for energy or converts into starch for storage.
Certain plants have evolved alternative photosynthetic pathways to cope with hot, dry climates. C4 plants, such as corn and sugarcane, spatially separate the initial carbon fixation step from the Calvin cycle to minimize water loss. CAM plants, like cacti, instead separate these processes by time, opening their stomata only at night to reduce evaporation while still collecting carbon dioxide.

The French Revolution began in 1789, driven by widespread financial crisis, food shortages, and deep resentment toward the rigid social hierarchy of the Ancien Régime. The Estates-General was convened in May 1789 for the first time in over a century, as King Louis XVI sought new taxes to address the state's crippling debt. Tensions between the three estates quickly escalated, culminating in the formation of the National Assembly by representatives of the Third Estate.
The storming of the Bastille on July 14, 1789 became a defining symbol of the Revolution. The prison held few political prisoners at the time, but it represented royal tyranny and served as a source of weapons and gunpowder for the revolutionaries. News of the event spread rapidly through France, triggering uprisings in the countryside known as the Great Fear, during which peasants attacked manor houses and destroyed feudal records.
The Reign of Terror, lasting from 1793 to 1794, saw the revolutionary government under Robespierre execute thousands of perceived enemies of the Revolution using the guillotine. The Committee of Public Safety wielded near-total power during this period, justifying mass executions as necessary to protect the Revolution from internal and external threats. Robespierre himself was eventually arrested and executed in July 1794, effectively ending the Terror.
Napoleon Bonaparte rose to power in the aftermath of the Revolution's instability, staging a coup in 1799 to establish the Consulate. He would later crown himself Emperor in 1804, effectively ending the revolutionary republic while retaining many of its legal and administrative reforms. The Napoleonic Code, established during his rule, standardized civil law across France and influenced legal systems throughout Europe.

HTTP, or Hypertext Transfer Protocol, is the foundational protocol used for communication on the World Wide Web. It operates as a request-response protocol between clients, typically web browsers, and servers. Each HTTP request includes a method such as GET, POST, PUT, or DELETE, indicating the action the client wants the server to perform on a given resource.
HTTP status codes are three-digit numbers returned by the server to indicate the outcome of a request. Codes in the 200 range indicate success, with 200 OK being the most common. The 300 range indicates redirection, the 400 range indicates a client-side error such as 404 Not Found, and the 500 range indicates a server-side error such as 500 Internal Server Error.
HTTP is a stateless protocol, meaning each request is processed independently without any memory of previous requests. To maintain state across multiple requests, such as keeping a user logged in, servers commonly use cookies, which are small pieces of data stored on the client and sent with each subsequent request. Session tokens and authentication headers serve similar purposes in modern web applications.
HTTPS extends HTTP by adding a layer of encryption via TLS, short for Transport Layer Security. This encryption protects data in transit from eavesdropping and tampering, which is especially important when transmitting sensitive information like passwords or payment details. Most modern browsers now flag plain HTTP sites as "not secure" to encourage widespread adoption of HTTPS.

Orders placed through our online store can be returned within 30 days of delivery for a full refund, provided the item is unused and in its original packaging. Refunds are processed to the original payment method within 5 to 7 business days after we receive the returned item. Customers are responsible for return shipping costs unless the item arrived damaged or defective.
Items marked as final sale, including clearance products and gift cards, are not eligible for return or exchange under any circumstances. To initiate a return, customers must first request a return authorization number through their account dashboard before shipping the item back. Returns sent without this authorization number may be refused or significantly delayed in processing.
For defective or damaged items, customers have 60 days from the delivery date to request a replacement or full refund, including return shipping costs. Photographic evidence of the damage is required before a replacement is shipped. Defective electronics may alternatively be exchanged directly at any physical retail location without needing to ship the item back first.
Exchanges for a different size or color are treated as a separate transaction from returns. Customers must return the original item for a refund and place a new order for the desired size or color, since the company does not process direct one-to-one exchanges. During high-demand sale periods, this can occasionally result in the desired replacement item selling out before the new order is placed.

A black hole is a region of spacetime where gravity is so strong that nothing, not even light, can escape once it crosses the event horizon. Black holes form when massive stars collapse under their own gravity at the end of their life cycle, typically after a supernova explosion. The point at the center of a black hole, where density becomes theoretically infinite, is called a singularity.
The event horizon marks the boundary beyond which escape from a black hole becomes impossible. Its size, known as the Schwarzschild radius, depends directly on the mass of the black hole. For a black hole with the mass of the Sun, the Schwarzschild radius would be approximately 3 kilometers, though stellar black holes typically have masses several times greater than the Sun.
Supermassive black holes, containing millions or even billions of solar masses, are believed to exist at the center of most large galaxies, including our own Milky Way. The supermassive black hole at the center of the Milky Way, named Sagittarius A*, was directly imaged for the first time in 2022 using the Event Horizon Telescope. Its existence had previously been inferred through the orbital motion of nearby stars.
Hawking radiation is a theoretical process through which black holes slowly lose mass and energy over immense timescales, eventually evaporating entirely. This phenomenon arises from quantum effects occurring near the event horizon. Because the rate of Hawking radiation is inversely related to a black hole's mass, smaller black holes are predicted to evaporate far faster than supermassive ones.

"""
para_ids = [
    "photosynthesis_1",
    "photosynthesis_2",
    "photosynthesis_3",
    "photosynthesis_4",
    "french_revolution_1",
    "french_revolution_2",
    "french_revolution_3",
    "french_revolution_4",
    "http_1",
    "http_2",
    "http_3",
    "http_4",
    "return_policy_1",   # standard 30-day return
    "return_policy_2",   # final sale exclusions
    "return_policy_3",   # defective/damaged — 60 days
    "return_policy_4",   # exchanges for size/color
    "black_holes_1",
    "black_holes_2",
    "black_holes_3",
    "black_holes_4",
]

In [ ]:
docs = corpus.split('\n')
docs

In [4]:
pre_pro_docs = [doc for doc in docs if doc.strip()]

In [5]:
len(pre_pro_docs)

20

In [ ]:
pre_pro_docs

In [25]:
hash_map_docId_to_name = {}
hash_map_name_to_docId = {}
for doc_id,doc_name in enumerate(para_ids):
  hash_map_docId_to_name[doc_id] = doc_name
  hash_map_name_to_docId[doc_name] = doc_id




In [30]:
# Check the Integrity of the hash_map_name_to_docId
def check_hash_map(dicts,docs):
  for doc_name in dicts :
    print(docs[dicts[doc_name]])
    print("\n NEXT DOCS \n")

In [32]:
# check_hash_map(hash_map_name_to_docId,pre_pro_docs)

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [8]:
tokenizer = model.tokenizer


In [12]:
# model.encode takes in a LIST OF STRINGS

def embed_corpus(model, docs,isnorm = True):
  embeddings = model.encode(docs,normalize_embeddings=isnorm)

  return embeddings

In [47]:
# # Test Function 1 (Embed Corpus)
embeds = embed_corpus(model,pre_pro_docs)
embeds.shape
# # Test Function 1 (Embed Corpus) quality of vectors
# np.linalg.norm(embeds[3])
# embeds.shape

(20, 384)

In [65]:
# This func requires input as string which then is internally converted to a list of string
# This string to list conversion is problematic and breaking the logic downstream, so directly use the query in the model.encode functions
# model.encode([query ]) --> Output is a numpy ndarray of size (1,384)
# model.encode(query) --> Output is a numpy ndarray of size (384,) which is important downstream for the optimized_scoring matrix multiplication

def embed_query(model,raw_query,isnorm = True):
  embeddings = model.encode(raw_query,normalize_embeddings=isnorm)

  return embeddings

In [68]:
# # Test Function 2 (Embed Query)
que = embed_query(model,'What is the plant based document')
que.shape
# que.shape
# np.linalg.norm(que)

(384,)

In [109]:
# This function needs The Array of the Docs Vectors and an Array of the Query Vector and converts score into float
def naive_loop_based_scoring(doc_vecs,query_vec):
  scores = []
  for vec in doc_vecs:
    scores.append(float(np.dot(query_vec,vec)))

  return scores

In [ ]:
# Test Function 3 (naive_loop_based_scoring)
sc = naive_loop_based_scoring(embeds,que)
sc

In [110]:
def optimized_scoring(doc_vecs,query_vec):
  return doc_vecs @ query_vec

In [82]:
# Test Function 4 (optimized_based_scoring)
a = optimized_scoring(embeds,que)
a

array([ 0.28807187,  0.2981574 ,  0.20720005,  0.34748685, -0.00035738,
       -0.02380253, -0.08576205,  0.07255034,  0.08117065,  0.0446791 ,
        0.03606916, -0.00163511, -0.00655474,  0.02595326, -0.06550778,
        0.036948  , -0.02221238, -0.12775354, -0.02140146, -0.14912345],
      dtype=float32)

In [122]:
# This function needs the score numpy array to work and it returns 2 array the ordered top_k_indices (index of the highest k scores) and top_k_scores actual scores
# If K > len(scores) this can be problematic and requires a safe limit and is done via min() funcs
# np.argsort returns a list of indices whose values/scores are ordered in ascending order. This function sorts the full list O(NlogN)
# np.argpartition function is similar to np.argsort but instead of full sort uses introselect as ver of quickselect
# to retrieve the top k elements and doesnt care about remaning elems O(N)
# This func I am using argsort and returning a list of top_k_scores that can be done via np.array accessing elems through a list of indices given instead of looping
def naive_rank_scores_return_doc_ids(scores,k = 5):
  scores = np.array(scores)
  if k <= 0:
    return np.array([], dtype=int), np.array([])

  k = min(k,len(scores))
  top_k_indices = np.argsort(scores)[::-1][:k]
  top_k_scores = np.array([scores[idx] for idx in top_k_indices])

  return top_k_indices , top_k_scores



In [119]:
# Test Function 5 (naive_rank_scores_return_doc_ids)
aa,b = naive_rank_scores_return_doc_ids(a)
aa,b

(array([3, 1, 0, 2, 8]),
 array([0.34748685, 0.2981574 , 0.28807187, 0.20720005, 0.08117065],
       dtype=float32))

In [107]:
# This is the optimized rank scoring fun
def optimized_rank_scores_return_doc_ids(scores,k = 5):
  if k <= 0:
    return np.array([], dtype=int), np.array([])

  scores = np.array(scores)
  k = min(k,len(scores))

  unsorted_top_k = np.argpartition(scores , -k)[-k:]
  top_k_indices = unsorted_top_k[np.argsort(-scores[unsorted_top_k])]
  top_k_scores = np.array(scores)[top_k_indices]

  return top_k_indices , top_k_scores




In [108]:
# Test Function 6 (optimized_rank_scores_return_doc_ids)
cc,d = optimized_rank_scores_return_doc_ids(a)
cc,d

(array([3, 1, 0, 2, 8]),
 array([0.34748685, 0.2981574 , 0.28807187, 0.20720005, 0.08117065],
       dtype=float32))

In [136]:
def retrieve(raw_q,doc_vecs,model,hash_map_docId_to_name,pre_pro_docs,k = 5,isnorm = True,opt_scoring = True,opt_ranking = True):
  if len(raw_q) == 0 : return None

  que_vec = embed_query(model,raw_q,isnorm)

  if opt_scoring :
    scores = optimized_scoring(doc_vecs,que_vec)
  else :
    scores = naive_loop_based_scoring(doc_vecs,que_vec)


  if opt_ranking :
    indices,top_scores = optimized_rank_scores_return_doc_ids(scores,k)
  else:
    indices,top_scores = naive_rank_scores_return_doc_ids(scores,k)

  result = []

  for doc_id , score in zip(indices,top_scores):
    result.append({
        'Doc_id' : int(doc_id),
        'Doc_name' : hash_map_docId_to_name[int(doc_id)],
        'Score' : float(score),
        'Text' : pre_pro_docs[int(doc_id)]
    })

  return result





In [128]:
# Test Function 7 (retreive)
res = retrieve('What is plant based document',embeds,model,hash_map_docId_to_name,pre_pro_docs)
res


[{'Doc_id': 3,
  'Doc_name': 'photosynthesis_4',
  'Score': 0.33479487895965576,
  'Text': 'Certain plants have evolved alternative photosynthetic pathways to cope with hot, dry climates. C4 plants, such as corn and sugarcane, spatially separate the initial carbon fixation step from the Calvin cycle to minimize water loss. CAM plants, like cacti, instead separate these processes by time, opening their stomata only at night to reduce evaporation while still collecting carbon dioxide.'},
 {'Doc_id': 0,
  'Doc_name': 'photosynthesis_1',
  'Score': 0.29833224415779114,
  'Text': 'Photosynthesis is the process by which plants convert light energy into chemical energy. It occurs primarily in the chloroplasts, specifically within structures called thylakoids. Chlorophyll, the green pigment in plants, absorbs light most efficiently in the blue and red wavelengths while reflecting green light. The overall reaction converts carbon dioxide and water into glucose and oxygen using energy from sunli

In [123]:
def build_prompt(raw_query,results):
  context = "\n\n".join(r['Text'] for r in results)

  prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain the answer, say so — do not use outside knowledge.
Context:
{context}
Question: {raw_query}
Answer:"""

  return prompt

In [124]:
# Test Function 8 (build prompt)
prompt = build_prompt('What is plant based document',res)
prompt

"Answer the question using ONLY the context below. If the context doesn't contain the answer, say so — do not use outside knowledge.\nContext:\nCertain plants have evolved alternative photosynthetic pathways to cope with hot, dry climates. C4 plants, such as corn and sugarcane, spatially separate the initial carbon fixation step from the Calvin cycle to minimize water loss. CAM plants, like cacti, instead separate these processes by time, opening their stomata only at night to reduce evaporation while still collecting carbon dioxide.\n\nPhotosynthesis is the process by which plants convert light energy into chemical energy. It occurs primarily in the chloroplasts, specifically within structures called thylakoids. Chlorophyll, the green pigment in plants, absorbs light most efficiently in the blue and red wavelengths while reflecting green light. The overall reaction converts carbon dioxide and water into glucose and oxygen using energy from sunlight.\n\nThe light-dependent reactions of

In [125]:
a = len(tokenizer.encode(prompt))
a

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (443 > 256). Running this sequence through the model will result in indexing errors


443

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="API_KEY"
)

def generate_answer(prompt, client, model_name="openai/gpt-oss-20b"):
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [127]:
answer = generate_answer(prompt, client)
print(answer)

I’m sorry, but the provided context does not contain any information about a “plant based document.”


In [131]:
## COMPLETE PIPELINE TEST
res = retrieve("How do plants convert sunlight into energy?", embeds, model, hash_map_docId_to_name, pre_pro_docs, k=4)
for r in res:
    print(f"[{r['Doc_name']}] {r['Score']:.3f}")

prompt = build_prompt("How do plants convert sunlight into energy?", res)
answer = generate_answer(prompt, client)
print(answer)

[photosynthesis_1] 0.757
[photosynthesis_2] 0.628
[photosynthesis_3] 0.589
[photosynthesis_4] 0.558
Plants convert sunlight into energy through photosynthesis. In the light‑dependent reactions that occur in the thylakoid membrane, chlorophyll absorbs light (especially blue and red wavelengths), splits water molecules, and uses the energy to produce ATP and NADPH. These high‑energy molecules then fuel the Calvin cycle in the stroma, where carbon dioxide is fixed into glucose. Thus, sunlight is transformed into chemical energy stored in glucose.


In [133]:
# Complete One Function for entire pipeline

def RAG_Complete(embeds, raw_query, model, hash_map_docId_to_name, pre_pro_docs, client,
                  k=5, isnorm=True, opt_scoring=True, opt_ranking=True, generate=True):

    res = retrieve(raw_query, embeds, model, hash_map_docId_to_name, pre_pro_docs,
                    k=k, isnorm=isnorm, opt_scoring=opt_scoring, opt_ranking=opt_ranking)

    if res is None:
        print("Empty query — nothing to retrieve.")
        return None

    print(f"Query: {raw_query}\n")
    for r in res:
        print(f"[{r['Doc_name']}] score={r['Score']:.3f}")

    prompt = build_prompt(raw_query, res)
    print(f"\n--- Prompt sent to model ---\n{prompt}\n")

    answer = None
    if generate:
        answer = generate_answer(prompt, client)
        print(f"--- Generated Answer ---\n{answer}")

    return {
        'results': res,
        'prompt': prompt,
        'answer': answer
    }

In [145]:
ans = RAG_Complete(
    embeds=embeds,
    raw_query="when did french revolution happend?",
    model=model,
    hash_map_docId_to_name=hash_map_docId_to_name,
    pre_pro_docs=pre_pro_docs,
    client=client,
    k=4
)

Query: when did french revolution happend?

[french_revolution_1] score=0.580
[french_revolution_2] score=0.518
[french_revolution_4] score=0.489
[french_revolution_3] score=0.346

--- Prompt sent to model ---
Answer the question using ONLY the context below. If the context doesn't contain the answer, say so — do not use outside knowledge.
Context:
The French Revolution began in 1789, driven by widespread financial crisis, food shortages, and deep resentment toward the rigid social hierarchy of the Ancien Régime. The Estates-General was convened in May 1789 for the first time in over a century, as King Louis XVI sought new taxes to address the state's crippling debt. Tensions between the three estates quickly escalated, culminating in the formation of the National Assembly by representatives of the Third Estate.

The storming of the Bastille on July 14, 1789 became a defining symbol of the Revolution. The prison held few political prisoners at the time, but it represented royal tyranny